# Cours 3 : Web Scraping avec BeautifulSoup - CORRIGÉ

## Objectifs
- Apprendre à scraper des sites web avec Beautiful Soup
- Extraire et sauvegarder des données depuis le web
- Utiliser la syntaxe moderne de BeautifulSoup (`find_all` au lieu de `findAll`)

In [7]:
# Imports nécessaires
import requests
from bs4 import BeautifulSoup
import pandas as pd
import random

---
# Partie 1 : Récupérer les liens vers tous les pays

**Objectif** : Scraper la page Worldometers pour obtenir la liste de tous les pays avec leurs liens.

**URL cible** : `https://www.worldometers.info/geography/how-many-countries-are-there-in-the-world/`

---
## Étape 1.1 : Se connecter au site

**Ce qu'il faut faire :**
1. Définir l'URL de la page à scraper
2. Créer un dictionnaire `headers` avec un User-Agent (pour éviter d'être bloqué)
3. Utiliser `requests.get()` pour récupérer la page
4. Vérifier que la connexion a réussi avec `response.ok`

**Conseil** : Un User-Agent simule un vrai navigateur.

In [8]:
# Étape 1.1 : Se connecter au site

url = 'https://www.worldometers.info/geography/how-many-countries-are-there-in-the-world/'

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

response = requests.get(url, headers=headers)

if response.ok:
    print('Connexion réussie !')
else:
    print(f'Erreur : Code {response.status_code}')

Connexion réussie !


---
## Étape 1.2 : Parser le HTML avec BeautifulSoup

**Ce qu'il faut faire :**
1. Créer un objet BeautifulSoup à partir de `response.text`
2. Utiliser le parser `'lxml'` (plus rapide et robuste)
3. Afficher un extrait du HTML pour vérifier

In [9]:
# Étape 1.2 : Parser le HTML

soup = BeautifulSoup(response.text, 'lxml')

# Afficher les 500 premiers caractères pour vérifier
print(soup.prettify()[:500])

<!DOCTYPE html>
<html data-astro-transition-scope="astro-s7reefwf-2" dir="ltr" lang="en">
 <head>
  <meta charset="utf-8"/>
  <meta content="IE=edge" http-equiv="X-UA-Compatible"/>
  <meta content="width=device-width, initial-scale=1" name="viewport"/>
  <title>
   How many countries are there in the world? (2026) - Total &amp; List - Worldometer
  </title>
  <!-- <I18nTags /> -->
  <script async="" src="https://www.googletagmanager.com/gtag/js?id=G-ZDP3BFSX60">
  </script>
  <script>
   window.


---
## Étape 1.3 : Trouver la table contenant les pays

**Ce qu'il faut faire :**
1. Utiliser l'inspecteur du navigateur pour identifier la table
2. La table a la classe `'data-table'`
3. Utiliser `soup.find()` avec le bon sélecteur

**Syntaxe moderne** : `soup.find('table', class_='data-table')`

In [15]:
# Étape 1.3 : Trouver la table

# Méthode 1 : avec class_ (syntaxe moderne)
table = soup.find('table', class_='datatable')

# Méthode 2 alternative : avec un dictionnaire (ancienne syntaxe, toujours valide)
# table = soup.find('table', {'class': 'data-table'})

if table:
    print('Table trouvée !')
else:
    print('Table non trouvée...')

Table trouvée !


---
## Étape 1.4 : Extraire tous les liens `<a>` de la table

**Ce qu'il faut faire :**
1. Utiliser `find_all('a')` sur la table (pas `findAll` !)
2. Compter le nombre de liens trouvés
3. Afficher le premier lien pour vérifier la structure

In [16]:
# Étape 1.4 : Extraire les liens

# Syntaxe moderne : find_all (avec underscore)
liens = table.find_all('a')

print(f'Nombre de liens trouvés : {len(liens)}')
print(f'\nPremier lien :')
print(liens[0].prettify())

Nombre de liens trouvés : 195

Premier lien :
<a class="transition-all duration-200 text-lime-600 hover:text-lime-500" data-astro-reload="true" href="/world-population/india-population/">
 India
</a>



---
## Étape 1.5 : Construire les URLs complètes et les noms des pays

**Ce qu'il faut faire :**
1. Créer deux listes vides : `liste_pays` et `liste_liens`
2. Boucler sur chaque lien `<a>`
3. Pour chaque lien :
   - Récupérer le texte avec `.get_text().strip()`
   - Récupérer l'URL avec `lien['href']`
   - Construire l'URL complète en ajoutant `'https://www.worldometers.info'`
4. Ajouter aux listes

In [17]:
# Étape 1.5 : Boucler et extraire les données

liste_pays = []
liste_liens = []

for lien in liens:
    # Récupérer le nom du pays (texte du lien)
    nom_pays = lien.get_text(strip=True)  # strip=True enlève les espaces
    
    # Récupérer l'URL relative
    url_relative = lien['href']
    
    # Construire l'URL complète
    url_complete = 'https://www.worldometers.info' + url_relative
    
    # Ajouter aux listes
    liste_pays.append(nom_pays)
    liste_liens.append(url_complete)

# Vérifier les résultats
print(f'Nombre de pays : {len(liste_pays)}')
print(f'\nPremiers pays :')
for i in range(5):
    print(f'  - {liste_pays[i]} : {liste_liens[i]}')

Nombre de pays : 195

Premiers pays :
  - India : https://www.worldometers.info/world-population/india-population/
  - China : https://www.worldometers.info/world-population/china-population/
  - United States : https://www.worldometers.info/world-population/us-population/
  - Indonesia : https://www.worldometers.info/world-population/indonesia-population/
  - Pakistan : https://www.worldometers.info/world-population/pakistan-population/


---
## Étape 1.6 : Créer un DataFrame et sauvegarder en CSV

**Ce qu'il faut faire :**
1. Créer un DataFrame pandas avec les colonnes `'pays'` et `'lien'`
2. Afficher les premières lignes avec `.head()`
3. Sauvegarder en CSV avec `.to_csv()`

In [18]:
# Étape 1.6 : Créer le DataFrame et sauvegarder

df_pays = pd.DataFrame({
    'pays': liste_pays,
    'lien': liste_liens
})

# Afficher les premières lignes
print('Aperçu du DataFrame :')
print(df_pays.head(10))

# Sauvegarder en CSV
df_pays.to_csv('liste_pays.csv', index=False)
print(f'\nFichier "liste_pays.csv" créé avec {len(df_pays)} pays !')

Aperçu du DataFrame :
            pays                                               lien
0          India  https://www.worldometers.info/world-population...
1          China  https://www.worldometers.info/world-population...
2  United States  https://www.worldometers.info/world-population...
3      Indonesia  https://www.worldometers.info/world-population...
4       Pakistan  https://www.worldometers.info/world-population...
5        Nigeria  https://www.worldometers.info/world-population...
6         Brazil  https://www.worldometers.info/world-population...
7     Bangladesh  https://www.worldometers.info/world-population...
8         Russia  https://www.worldometers.info/world-population...
9       Ethiopia  https://www.worldometers.info/world-population...

Fichier "liste_pays.csv" créé avec 195 pays !


---
## Résumé Partie 1 : Code complet

**Objectif** : Rassembler toutes les étapes en un seul bloc de code.

In [20]:
# RÉSUMÉ PARTIE 1 : Code complet pour récupérer tous les liens des pays

import requests
from bs4 import BeautifulSoup
import pandas as pd

# 1. Connexion
url = 'https://www.worldometers.info/geography/how-many-countries-are-there-in-the-world/'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
response = requests.get(url, headers=headers)

# 2. Parsing
soup = BeautifulSoup(response.text, 'lxml')

# 3. Trouver la table
table = soup.find('table', class_='datatable')

# 4. Extraire les liens
liens = table.find_all('a')

# 5. Construire les listes
liste_pays = [lien.get_text(strip=True) for lien in liens]
liste_liens = ['https://www.worldometers.info' + lien['href'] for lien in liens]

# 6. Créer et sauvegarder le DataFrame
df_pays = pd.DataFrame({'pays': liste_pays, 'lien': liste_liens})
df_pays.to_csv('liste_pays.csv', index=False)

print(f'Terminé ! {len(df_pays)} pays récupérés.')
df_pays.head(10)

Terminé ! 195 pays récupérés.


,pays,lien
0,India,https://www.worldometers.info/world-population...
1,China,https://www.worldometers.info/world-population...
2,United States,https://www.worldometers.info/world-population...
3,Indonesia,https://www.worldometers.info/world-population...
4,Pakistan,https://www.worldometers.info/world-population...
5,Nigeria,https://www.worldometers.info/world-population...
6,Brazil,https://www.worldometers.info/world-population...
7,Bangladesh,https://www.worldometers.info/world-population...
8,Russia,https://www.worldometers.info/world-population...
9,Ethiopia,https://www.worldometers.info/world-population...


---
# Partie 2 : Récupérer la population prévue en 2035 pour un pays aléatoire

**Objectif** : Choisir un pays au hasard et récupérer sa population prévue pour 2035.

---
## Étape 2.1 : Choisir un pays au hasard

**Ce qu'il faut faire :**
1. Utiliser `random.choice()` pour sélectionner un pays au hasard dans `liste_pays`
2. Récupérer l'index de ce pays
3. Récupérer le lien correspondant dans `liste_liens`
4. Afficher le pays choisi et son lien

In [21]:
# Étape 2.1 : Choisir un pays au hasard

import random

# Choisir un index aléatoire
index_aleatoire = random.randint(0, len(liste_pays) - 1)

# Récupérer le pays et son lien
pays_choisi = liste_pays[index_aleatoire]
lien_pays = liste_liens[index_aleatoire]

print(f'Pays choisi : {pays_choisi}')
print(f'Lien : {lien_pays}')

Pays choisi : China
Lien : https://www.worldometers.info/world-population/china-population/


---
## Étape 2.2 : Se connecter à la page du pays

**Ce qu'il faut faire :**
1. Utiliser `requests.get()` avec le lien du pays
2. Ne pas oublier les headers !
3. Vérifier que la connexion a réussi

In [22]:
# Étape 2.2 : Se connecter à la page du pays

response_pays = requests.get(lien_pays, headers=headers)

if response_pays.ok:
    print(f'Connexion à la page de {pays_choisi} réussie !')
else:
    print(f'Erreur : Code {response_pays.status_code}')

Connexion à la page de China réussie !


---
## Étape 2.3 : Parser le HTML de la page du pays

**Ce qu'il faut faire :**
1. Créer un objet BeautifulSoup
2. Afficher un extrait pour explorer la structure

In [23]:
# Étape 2.3 : Parser le HTML

soup_pays = BeautifulSoup(response_pays.text, 'lxml')

print('Page parsée avec succès !')

Page parsée avec succès !


---
## Étape 2.4 : Trouver la table des projections de population

**Ce qu'il faut faire :**
1. La page contient plusieurs tables
2. Chercher toutes les tables avec `find_all('table')`
3. La table des projections a aussi la classe `'data-table'`
4. Explorer les tables pour trouver celle avec les projections (années futures)

In [29]:
# Étape 2.4 : Trouver la table des projections

# Récupérer toutes les tables
toutes_tables = soup_pays.find_all('table', class_='datatable')
print(f'Nombre de tables trouvées : {len(toutes_tables)}')

# La table des projections est généralement la 2ème ou 3ème table
# On va chercher celle qui contient des années futures (>2025)
table_projections = None

for i, t in enumerate(toutes_tables):
    # Regarder le contenu de la première cellule
    premiere_ligne = t.find('tr')
    if premiere_ligne:
        texte = premiere_ligne.get_text()
        # Si on trouve "2035" ou des années futures, c'est probablement la bonne
        if '2035' in t.get_text() or '2030' in t.get_text():
            table_projections = t
            print(f'Table des projections trouvée (index {i}) !')
            break

if not table_projections:
    print('Table des projections non trouvée, utilisation de la dernière table')
    table_projections = toutes_tables[-1] if toutes_tables else None

Nombre de tables trouvées : 3
Table des projections trouvée (index 1) !


---
## Étape 2.5 : Extraire toutes les lignes de la table

**Ce qu'il faut faire :**
1. Utiliser `find_all('tr')` pour récupérer toutes les lignes
2. La première ligne est l'en-tête (à ignorer)
3. Afficher une ligne pour voir sa structure

In [30]:
# Étape 2.5 : Extraire les lignes

if table_projections:
    lignes = table_projections.find_all('tr')
    print(f'Nombre de lignes : {len(lignes)}')
    
    # Afficher la structure d'une ligne (par exemple la 2ème, index 1)
    if len(lignes) > 1:
        print(f'\nExemple de ligne :')
        print(lignes[1].prettify())

Nombre de lignes : 6

Exemple de ligne :
<tr>
 <td class="px-2 border-e border-zinc-200 text-end py-1.5 border-b" data-order="20300000">
  2030
 </td>
 <td class="px-2 border-e border-zinc-200 text-end py-1.5 border-b font-bold" data-order="13981538320000" dir="ltr">
  1,398,153,832
 </td>
 <td class="px-2 border-e border-zinc-200 text-end py-1.5 border-b" data-order="-25" dir="ltr">
  â0.25%
 </td>
 <td class="px-2 border-e border-zinc-200 text-end py-1.5 border-b" data-order="-35884524000" dir="ltr">
  â3,588,452
 </td>
 <td class="px-2 border-e border-zinc-200 text-end py-1.5 border-b" data-order="-1991920000" dir="ltr">
  â199,192
 </td>
 <td class="px-2 border-e border-zinc-200 text-end py-1.5 border-b" data-order="429160" dir="ltr">
  42.9
 </td>
 <td class="px-2 border-e border-zinc-200 text-end py-1.5 border-b" data-order="10610" dir="ltr">
  1.06
 </td>
 <td class="px-2 border-e border-zinc-200 text-end py-1.5 border-b" data-order="1489265" dir="ltr">
  149
 </td>
 <td c

---
## Étape 2.6 : Trouver la ligne correspondant à 2035

**Ce qu'il faut faire :**
1. Boucler sur les lignes (en sautant l'en-tête)
2. Pour chaque ligne, récupérer les cellules avec `find_all('td')`
3. La première cellule contient l'année
4. Si l'année est '2035', récupérer la deuxième cellule (population)

In [31]:
# Étape 2.6 : Trouver la ligne 2035

population_2035 = None

if table_projections:
    for ligne in lignes[1:]:  # On saute l'en-tête (index 0)
        cellules = ligne.find_all('td')
        
        if len(cellules) >= 2:
            # Première cellule = année
            annee = cellules[0].get_text(strip=True)
            print('year :',annee)
            
            if '2035' in annee:
                # Deuxième cellule = population
                population_2035 = cellules[1].get_text(strip=True)
                print(f'Ligne 2035 trouvée !')
                print(f'Année : {annee}')
                print(f'Population : {population_2035}')
                break

if not population_2035:
    print('Année 2035 non trouvée dans la table')

year : 2030
year : 2035
Ligne 2035 trouvée !
Année : 2035
Population : 1,373,427,531


---
## Étape 2.7 : Afficher le résultat

**Ce qu'il faut faire :**
1. Afficher le nom du pays
2. Afficher la population prévue en 2035
3. Nettoyer le texte avec `.strip()`

In [32]:
# Étape 2.7 : Afficher le résultat

print('=' * 50)
print(f'RÉSULTAT')
print('=' * 50)
print(f'Pays : {pays_choisi}')
print(f'Population prévue en 2035 : {population_2035}')
print('=' * 50)

RÉSULTAT
Pays : China
Population prévue en 2035 : 1,373,427,531


---
## Résumé Partie 2 : Code complet

**Objectif** : Rassembler toutes les étapes en un seul bloc de code.

In [34]:
# RÉSUMÉ PARTIE 2 : Code complet pour récupérer la population 2035 d'un pays aléatoire

import random

# 1. Choisir un pays aléatoire (suppose que liste_pays et liste_liens existent)
index_aleatoire = random.randint(0, len(liste_pays) - 1)
pays_choisi = liste_pays[index_aleatoire]
lien_pays = liste_liens[index_aleatoire]

# 2. Connexion à la page du pays
response_pays = requests.get(lien_pays, headers=headers)
soup_pays = BeautifulSoup(response_pays.text, 'lxml')

# 3. Trouver la table des projections
toutes_tables = soup_pays.find_all('table', class_='datatable')
table_projections = None
for t in toutes_tables:
    if '2035' in t.get_text():
        table_projections = t
        break

# 4. Chercher la ligne 2035
population_2035 = 'Non trouvée'
if table_projections:
    for ligne in table_projections.find_all('tr')[1:]:
        cellules = ligne.find_all('td')
        if len(cellules) >= 2 and '2035' in cellules[0].get_text():
            population_2035 = cellules[1].get_text(strip=True)
            break

# 5. Afficher le résultat
print(f'Pays : {pays_choisi}')
print(f'Population 2035 : {population_2035}')

Pays : Equatorial Guinea
Population 2035 : 2,414,269


---
# Partie 3 : Code final combiné

**Objectif** : Combiner les Parties 1 et 2 en un seul script complet qui :
1. Récupère tous les liens des pays
2. Choisit un pays au hasard
3. Récupère et affiche sa population prévue pour 2035

In [36]:
# CODE FINAL COMBINÉ

# Configuration
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

# PARTIE 1 : Récupérer tous les pays

print('=== PARTIE 1 : Récupération de la liste des pays ===')

url_liste = 'https://www.worldometers.info/geography/how-many-countries-are-there-in-the-world/'
response = requests.get(url_liste, headers=headers)

if not response.ok:
    raise Exception(f'Erreur de connexion : {response.status_code}')

soup = BeautifulSoup(response.text, 'lxml')
table = soup.find('table', class_='datatable')
liens = table.find_all('a')

liste_pays = [lien.get_text(strip=True) for lien in liens]
liste_liens = ['https://www.worldometers.info' + lien['href'] for lien in liens]

print(f'✓ {len(liste_pays)} pays récupérés')

# PARTIE 2 : Choisir un pays et obtenir 2035

print('\n=== PARTIE 2 : Sélection aléatoire et population 2035 ===')

# Choisir au hasard
index = random.randint(0, len(liste_pays) - 1)
pays_choisi = liste_pays[index]
lien_pays = liste_liens[index]

print(f'Pays sélectionné : {pays_choisi}')

# Récupérer la page du pays
response_pays = requests.get(lien_pays, headers=headers)
soup_pays = BeautifulSoup(response_pays.text, 'lxml')

# Trouver la table des projections
population_2035 = 'Non disponible'
for table in soup_pays.find_all('table', class_='datatable'):
    if '2035' in table.get_text():
        for ligne in table.find_all('tr')[1:]:
            cellules = ligne.find_all('td')
            if len(cellules) >= 2 and '2035' in cellules[0].get_text():
                population_2035 = cellules[1].get_text(strip=True)
                break
        break



=== PARTIE 1 : Récupération de la liste des pays ===
✓ 195 pays récupérés

=== PARTIE 2 : Sélection aléatoire et population 2035 ===
Pays sélectionné : Palau


---
# Récapitulatif des méthodes BeautifulSoup

| Méthode moderne | Ancienne syntaxe | Description |
|-----------------|------------------|-------------|
| `soup.find('tag')` | `soup.find('tag')` | Trouver le **premier** élément |
| `soup.find_all('tag')` | `soup.findAll('tag')` | Trouver **tous** les éléments |
| `soup.find('tag', class_='nom')` | `soup.find('tag', {'class': 'nom'})` | Filtrer par classe CSS |
| `element['href']` | `element['href']` | Récupérer un attribut |
| `element.get_text()` | `element.get_text()` | Récupérer le texte |
| `element.get_text(strip=True)` | `element.get_text().strip()` | Récupérer le texte nettoyé |